# ml_scan milestone gates

One markdown + one code cell per milestone. Restart-safe. Same checks as `user_command.md`.

**Date window:** this warehouse has hourly bars from **2025-09-01** onward, not 2024. Run the first code cell so `SMOKE_START` / `SMOKE_END` match live coverage, then continue from M09 if you already passed M01–M08.

In [1]:
from pathlib import Path
import pandas as pd
from ml_scan.config import load_settings
from ml_scan.storage.db import Database

ROOT = Path('.').resolve()
pd.set_option('display.max_columns', 40)

# Warehouse has no 2024 bars; derive the smoke window from live coverage.
db = Database.from_settings(load_settings())
with db.connection() as c:
    cov = c.execute('select min(ts) as mn, max(ts) as mx from ohlcv_60m').fetchone()
SMOKE_START = pd.Timestamp(cov['mn']).tz_convert('Asia/Kolkata').strftime('%Y-%m-%d')
SMOKE_END = pd.Timestamp(cov['mx']).tz_convert('Asia/Kolkata').strftime('%Y-%m-%d')
print(ROOT)
print('warehouse', SMOKE_START, '→', SMOKE_END)

C:\Users\mail2\OneDrive\projects2\ml_scan
warehouse 2025-09-01 → 2026-09-03


## M01 — Repo skeleton

Settings must load and Timescale DB name must be `scan_trade`.

In [2]:
import ml_scan
import ml_scan.config
print(ml_scan.config.load_settings().timescale.db)

scan_trade


## M02 — Storage copy

`ohlcv_60m` must already have rows in the existing warehouse.

In [3]:
from ml_scan.storage.db import Database
from ml_scan.config import load_settings
db = Database.from_settings(load_settings())
with db.connection() as c:
    n = c.execute('select count(*) as n from ohlcv_60m').fetchone()['n']
print('hourly_rows', n)
assert n > 0

hourly_rows 868999


## M03 — Calendar primitives

NSE hourly index is 7 bars per session.

In [4]:
import pandas as pd
from ml_scan.data.calendar import NSECalendar
c = NSECalendar()
print(len(c.hourly_index(pd.Timestamp('2024-01-02', tz='Asia/Kolkata'))))

7


## M04 — TimescaleAdapter

Hourly read for RELIANCE; no 15:15 partial hours.

In [5]:
from ml_scan.config import load_settings
from ml_scan.data.timescale_adapter import TimescaleAdapter
a = TimescaleAdapter(load_settings())
df = a.read(['RELIANCE'], '60minute', SMOKE_START, SMOKE_END)
assert len(df) > 0, f'no hourly bars for RELIANCE between {SMOKE_START} and {SMOKE_END}'
assert {'ts','symbol','open','high','low','close','volume','interval'} <= set(df.columns)
assert (df['interval'] == '60minute').all()
ts = pd.to_datetime(df['ts'], utc=True).dt.tz_convert('Asia/Kolkata')
assert not ts.dt.strftime('%H:%M').eq('15:15').any()
df.head()

,ts,symbol,instrument_token,open,high,low,close,volume,source,n_5m,interval,is_partial_hour
0,2025-09-01 09:15:00+05:30,RELIANCE,738561,1356.0,1357.0,1340.6,1351.3,2837205.0,cagg,12,60minute,False
1,2025-09-01 10:15:00+05:30,RELIANCE,738561,1351.4,1353.8,1350.8,1352.6,1323586.0,cagg,12,60minute,False
2,2025-09-01 11:15:00+05:30,RELIANCE,738561,1352.7,1353.5,1348.6,1353.0,1116376.0,cagg,12,60minute,False
3,2025-09-01 12:15:00+05:30,RELIANCE,738561,1353.0,1359.0,1351.7,1356.9,1550364.0,cagg,12,60minute,False
4,2025-09-01 13:15:00+05:30,RELIANCE,738561,1356.9,1363.2,1355.9,1360.1,1463012.0,cagg,12,60minute,False


## M05 — Universe snapshot

In [6]:
!python -m ml_scan.cli universe snapshot --source "C:\Users\mail2\OneDrive\projects2\scan_trade\data\universe\nifty500_mapped.csv"
import pandas as pd
u = pd.read_csv('data/universe/nifty500_mapped.csv')
print(len(u), u.columns.tolist())

wrote C:\Users\mail2\OneDrive\projects2\ml_scan\data\universe\nifty500_mapped.csv
501 ['symbol', 'isin', 'industry', 'company', 'series', 'instrument_token', 'exchange', 'asof_ist']


## M06 — Liquidity filter (ADTV > ₹5 Cr)

In [7]:
!python -m ml_scan.cli universe liquid --adtv-min 50000000 --smoke-n 8 --out data/universe/liquid_universe.csv
import pandas as pd
liq = pd.read_csv('data/universe/liquid_universe.csv')
print(liq[['symbol','adtv_20']].head())

liquid=C:\Users\mail2\OneDrive\projects2\ml_scan\data\universe\liquid_universe.csv smoke=C:\Users\mail2\OneDrive\projects2\ml_scan\data\universe\smoke_symbols.csv
       symbol       adtv_20
0    HDFCBANK  1.993416e+10
1         BSE  1.589829e+10
2    RELIANCE  1.419166e+10
3   ICICIBANK  1.337102e+10
4  BHARTIARTL  1.312065e+10


## M07 — MTFAligner

Print hourly `ts` vs joined `d_close` date. Same-session daily close must not leak.

In [8]:
import subprocess, sys
import pandas as pd

start, end = SMOKE_START, SMOKE_END
subprocess.check_call([
    sys.executable, "-m", "ml_scan.cli", "data", "align",
    "--symbols", "RELIANCE,TCS",
    "--start", start, "--end", end,
    "--out", "data/artifacts/align_smoke.parquet",
])
subprocess.check_call([sys.executable, "-m", "pytest", "tests/test_mtf_aligner.py", "-q"])
al = pd.read_parquet('data/artifacts/align_smoke.parquet')
print(al[['symbol','ts','d_ts','d_close']].head(8))
print(al.shape)

     symbol                        ts                      d_ts  d_close
0  RELIANCE 2025-09-01 09:15:00+05:30 2025-08-29 00:00:00+05:30   1357.2
1  RELIANCE 2025-09-01 10:15:00+05:30 2025-08-29 00:00:00+05:30   1357.2
2  RELIANCE 2025-09-01 11:15:00+05:30 2025-08-29 00:00:00+05:30   1357.2
3  RELIANCE 2025-09-01 12:15:00+05:30 2025-08-29 00:00:00+05:30   1357.2
4  RELIANCE 2025-09-01 13:15:00+05:30 2025-08-29 00:00:00+05:30   1357.2
5  RELIANCE 2025-09-01 14:15:00+05:30 2025-08-29 00:00:00+05:30   1357.2
6  RELIANCE 2025-09-02 09:15:00+05:30 2025-09-01 00:00:00+05:30   1353.9
7  RELIANCE 2025-09-02 10:15:00+05:30 2025-09-01 00:00:00+05:30   1353.9
(2992, 24)


## M08 — TA engine parity

In [9]:
!python -m pytest tests/test_ta_engine_parity.py -q

.                                                                        [100%]
============================== warnings summary ===============================
tests/test_ta_engine_parity.py::test_ta_engine_matches_legacy_helper
tests/test_ta_engine_parity.py::test_ta_engine_matches_legacy_helper
  c:\Users\mail2\OneDrive\projects2\ml_scan\.ml_env\Lib\site-packages\pandas\core\window\rolling.py:611: RuntimeWarning: All-NaN slice encountered
    return func(x, start, end, min_periods, *numba_args)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
1 passed, 2 warnings in 1.26s


## M09 — PanelFeatureEngineer

In [10]:
import subprocess, sys
import pandas as pd

start, end = SMOKE_START, SMOKE_END
subprocess.check_call([
    sys.executable, "-m", "ml_scan.cli", "features", "hourly",
    "--universe", "data/universe/smoke_symbols.csv",
    "--start", start, "--end", end,
    "--out", "data/artifacts/feat_hourly_smoke.parquet",
])
feat = pd.read_parquet('data/artifacts/feat_hourly_smoke.parquet')
assert len(feat) > 0, f'hourly features empty for {start} → {end}'
assert 'ATR' in feat.columns, 'TA columns missing'
meta = {'symbol', 'ts', 'interval', 'source', 'instrument_token', 'open', 'high', 'low', 'close', 'volume'}
n_ta = len([c for c in feat.columns if c not in meta])
print(f'hourly panel: {len(feat)} rows x {feat.shape[1]} columns ({n_ta} pandas_ta / derived, mode from configs/default.yaml)')
print(feat.columns[:20].tolist())
print(feat[['symbol','ts']].head())

hourly panel: 11968 rows x 239 columns (229 pandas_ta / derived, mode from configs/default.yaml)
['ts', 'open', 'high', 'low', 'close', 'volume', 'EBSW', 'REFLEX', 'AO', 'APO', 'BIAS', 'CCI', 'CFO', 'CG', 'CMO', 'COPPOCK', 'CRSI', 'CTI', 'ER', 'BULLP_13']
       symbol                        ts
0  BHARTIARTL 2025-09-01 09:15:00+05:30
1  BHARTIARTL 2025-09-01 10:15:00+05:30
2  BHARTIARTL 2025-09-01 11:15:00+05:30
3  BHARTIARTL 2025-09-01 12:15:00+05:30
4  BHARTIARTL 2025-09-01 13:15:00+05:30


### M09b — Which stocks feed the model, and how many pandas_ta features it adds (Q1, Q2)

**Q1**: the split used later (M15b) is time-based, not stock-based, so every symbol in `smoke_symbols.csv` is used identically for both training and testing — there is no separate "held-out stock" list. The table below is the per-symbol row count feeding the pipeline from this point on.

**Q2**: M09 now runs pandas_ta in **full** mode (`features.hourly_mode: full` in `configs/default.yaml`) — every compatible indicator, matching `ml_reference/1_feature_engineering.ipynb` (`generate_all_ta_features`, 200+ columns). The cell below also prints the lite-mode count so the size of that jump is visible. These are *hourly* TA columns only; M10 below adds daily and 15-minute context on top, then M12 QC drops low-quality columns (high missingness / zero variance, treating inf as missing, same idea as the reference notebook).

In [11]:
import pandas as pd

from ml_scan.config import load_settings
from ml_scan.data.universe import load_symbol_list
from ml_scan.features.qc import symbol_row_counts
from ml_scan.features.ta_engine import generate_all_ta_features, generate_lite_ta_features

symbols = load_symbol_list('data/universe/smoke_symbols.csv')
print(f"Q1 -- {len(symbols)} smoke-universe stocks feed this pipeline: {symbols}")

hourly = pd.read_parquet('data/artifacts/feat_hourly_smoke.parquet')
print("\nrows contributed per stock at the hourly-feature stage:")
display(symbol_row_counts(hourly)[['symbol', 'n_rows']])

meta = {'symbol', 'ts', 'interval', 'source', 'instrument_token', 'open', 'high', 'low', 'close', 'volume'}
n_in_panel = len([c for c in hourly.columns if c not in meta])
mode = load_settings().features.hourly_mode
print(f"\nQ2 -- configured hourly_mode={mode!r}: {n_in_panel} non-OHLCV columns actually written by M09")

sample = (
    hourly.loc[hourly['symbol'] == symbols[0], ['ts', 'open', 'high', 'low', 'close', 'volume']]
    .sort_values('ts')
    .set_index('ts')
)
ohlcv_cols = {'open', 'high', 'low', 'close', 'volume'}
n_lite = len(set(generate_lite_ta_features(sample).columns) - ohlcv_cols)
n_full = len(set(generate_all_ta_features(sample).columns) - ohlcv_cols)
print(f"Q2 -- pandas_ta 'lite' mode (small smoke set): {n_lite} new indicator columns")
print(f"Q2 -- pandas_ta 'full' mode (every compatible indicator): {n_full} new indicator columns")

Q1 -- 8 smoke-universe stocks feed this pipeline: ['HDFCBANK', 'BSE', 'RELIANCE', 'ICICIBANK', 'BHARTIARTL', 'ETERNAL', 'SBIN', 'NETWEB']

rows contributed per stock at the hourly-feature stage:


,symbol,n_rows
0,BHARTIARTL,1496
1,BSE,1496
2,ETERNAL,1496
3,HDFCBANK,1496
4,ICICIBANK,1496
5,NETWEB,1496
6,RELIANCE,1496
7,SBIN,1496



Q2 -- configured hourly_mode='full': 229 non-OHLCV columns actually written by M09
Q2 -- pandas_ta 'lite' mode (small smoke set): 22 new indicator columns
Q2 -- pandas_ta 'full' mode (every compatible indicator): 227 new indicator columns


## M10 — Daily + 15m join

In [12]:
!python -m ml_scan.cli features mtf --in data/artifacts/feat_hourly_smoke.parquet --out data/artifacts/feat_mtf_smoke.parquet
import pandas as pd
mtf = pd.read_parquet('data/artifacts/feat_mtf_smoke.parquet')
d_cols = [c for c in mtf.columns if c.startswith('d_')]
m15_cols = [c for c in mtf.columns if c.startswith('m15_')]
assert d_cols, 'daily columns missing'
assert m15_cols, '15-minute columns missing'
print(d_cols[:8])
print(m15_cols[:8])

wrote C:\Users\mail2\OneDrive\projects2\ml_scan\data\artifacts\feat_mtf_smoke.parquet
['d_ts', 'd_open', 'd_high', 'd_low', 'd_close', 'd_volume', 'd_ema50', 'd_ema200']
['m15_ts', 'm15_open', 'm15_high', 'm15_low', 'm15_close', 'm15_volume', 'm15_rsi', 'm15_vwap_dist']


C:\Users\mail2\OneDrive\projects2\ml_scan\src\ml_scan\features\htf.py:92: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(parts, ignore_index=True)


## M11 — SwingLabeler

Print class balance. The next cell (M11b) turns that imbalance into inverse-frequency class weights, matching `ml_reference/1_feature_engineering.ipynb`.

In [13]:
!python -m ml_scan.cli features label --in data/artifacts/feat_mtf_smoke.parquet --out data/artifacts/labeled_smoke.parquet
import pandas as pd
lab = pd.read_parquet('data/artifacts/labeled_smoke.parquet')
assert len(lab) > 0, 'labeled panel is empty — re-run M09/M10 with warehouse dates'
print(lab['y'].value_counts(dropna=False))
print(lab['y_reason'].value_counts(dropna=False))

y
0.0    7682
1.0    3958
NaN     328
y_reason
sl_hit        7682
tp_hit        3958
unresolved     328
wrote C:\Users\mail2\OneDrive\projects2\ml_scan\data\artifacts\labeled_smoke.parquet
y
0.0    7682
1.0    3958
NaN     328
Name: count, dtype: int64
y_reason
sl_hit        7682
tp_hit        3958
unresolved     328
Name: count, dtype: int64


### M11b — Class imbalance and class weights

The swing label is typically imbalanced (more stop-hits than take-profit hits). Training without weights makes trees predict the majority class and inflates raw accuracy while crushing recall — which is why the earlier smoke metrics looked worse than they needed to.

Weights follow the reference `cwts` formula: `w_i = n / (2 * n_i)`. With those weights, both classes contribute equally (`w0 * n0 == w1 * n1 == n/2`). From here on, RandomForest / LightGBM / XGBoost, Boruta, and hyperparameter search all use these weights (computed on the *train* fold only, never the test fold).

In [14]:
import pandas as pd

from ml_scan.ml_engine.metrics import class_weight_dict

lab = pd.read_parquet('data/artifacts/labeled_smoke.parquet')
resolved = lab.loc[lab['y'].isin([0, 1]), 'y'].astype(int)
counts = resolved.value_counts().sort_index()
print('resolved label counts:')
print(counts.to_string())
print(f"\npositive rate (wins): {resolved.mean():.4f}")

cw = class_weight_dict(resolved)
print('\nclass weights (cwts):')
print({k: round(v, 4) for k, v in cw.items()})
print(f"weighted class 0 contribution: {cw[0] * int(counts.get(0, 0)):.4f}")
print(f"weighted class 1 contribution: {cw[1] * int(counts.get(1, 0)):.4f}")
print('(those two should be equal — both classes now pull with the same total weight)')

resolved label counts:
y
0    7682
1    3958

positive rate (wins): 0.3400

class weights (cwts):
{0: 0.7576, 1: 1.4704}
weighted class 0 contribution: 5820.0000
weighted class 1 contribution: 5820.0000
(those two should be equal — both classes now pull with the same total weight)


## M12 — Feature QC + leakage

In [15]:
!python -m ml_scan.cli features qc --in data/artifacts/labeled_smoke.parquet --out data/artifacts/feature_qc_report.json
!python -m pytest tests/test_leakage.py -q

{
  "n_rows": 11968,
  "n_feature_cols": 218,
  "leakage_ok": false
}
wrote C:\Users\mail2\OneDrive\projects2\ml_scan\data\artifacts\feature_qc_report.json
..                                                                       [100%]
2 passed in 0.03s


### M12b — Baseline model comparison on the initial feature set (Q0, Q3)

Before Boruta ever runs, fit **RandomForest, XGBoost, and LightGBM** on every QC-passed feature from the **full** pandas_ta matrix, scored on the same purged walk-forward folds used everywhere below. Each model is trained with the M11b class weights (per-row `sample_weight` from `cwts`, recomputed on that fold's training labels only).

This is the first place `accuracy`, `balanced_accuracy`, `precision`, `recall`, and `roc_auc` are all reported together (Q0), for all three candidate models (Q3). The winner (by fold-weighted ROC-AUC) is written to `data/artifacts/model_comparison_initial.json` as `best_model`, and is the model the rest of this notebook's diagnostic cells (M13b, M14b, M16a, M16b) carry forward.

In [16]:
import json
from pathlib import Path

import pandas as pd

from ml_scan.config import load_settings
from ml_scan.features.qc import run_qc, select_xy
from ml_scan.ml_engine.estimator import compare_models, select_best_model
from ml_scan.ml_engine.metrics import fold_metrics_table
from ml_scan.ml_engine.splitter import PurgedWalkForward

settings = load_settings()
labeled = pd.read_parquet('data/artifacts/labeled_smoke.parquet')
_, qc_report = run_qc(labeled, missing_threshold=settings.features.missing_threshold)
initial_features = qc_report['features']
print(f"Q2/Q3 setup: {len(initial_features)} QC-passed features feed the baseline comparison")
print(initial_features)

X, y, panel = select_xy(labeled, initial_features)
splitter = PurgedWalkForward(n_splits=settings.ml.n_splits, embargo_sessions=settings.ml.embargo_sessions)
baseline_summary, baseline_models = compare_models(X, y, panel, splitter=splitter, random_state=settings.ml.random_state)
best_model = select_best_model(baseline_summary, metric='roc_auc')

print("\nQ0 / Q3 -- baseline modeling results, initial feature set, every model:")
display(baseline_summary.round(4))
print(f"\nbest model by fold-weighted ROC-AUC: {best_model!r}")
print("\nfold-by-fold detail for the winner:")
display(fold_metrics_table(baseline_models[best_model].fold_metrics_).round(4))

dest = Path('data/artifacts/model_comparison_initial.json')
dest.write_text(
    json.dumps(
        {
            'best_model': best_model,
            'n_initial_features': len(initial_features),
            'initial_features': initial_features,
            'summary': baseline_summary.reset_index().to_dict(orient='records'),
        },
        indent=2,
    ),
    encoding='utf-8',
)
print(f"\nwrote {dest}")

Q2/Q3 setup: 218 QC-passed features feed the baseline comparison
['EBSW', 'REFLEX', 'AO', 'APO', 'BIAS', 'CCI', 'CFO', 'CG', 'CMO', 'COPPOCK', 'CRSI', 'CTI', 'ER', 'BULLP_13', 'BEARP_13', 'EXHC_DNa', 'EXHC_UPa', 'FISHERT_9_1', 'FISHERTs_9_1', 'INERTIA', 'K_9_3', 'D_9_3', 'J_9_3', 'KST_10_15_20_30_10_10_10_15', 'KSTs_9', 'MACD_12_26_9', 'MACDh_12_26_9', 'MACDs_12_26_9', 'MOM', 'PGO', 'PPO_12_26_9', 'PPOh_12_26_9', 'PPOs_12_26_9', 'PSL', 'QQE_14_5_4.236', 'QQE_14_5_4.236_RSIMA', 'ROC', 'RSI', 'RSX', 'SLOPE', 'SMI_5_20_5_1.0', 'SMIs_5_20_5_1.0', 'SMIo_5_20_5_1.0', 'SQZ_20_2.0_20_1.5', 'SQZ_ON', 'SQZ_OFF', 'SQZ_NO', 'SQZPRO_20_2.0_20_2.0_1.5_1.0', 'SQZPRO_ON_WIDE', 'SQZPRO_ON_NORMAL', 'SQZPRO_ON_NARROW', 'SQZPRO_OFF', 'SQZPRO_NO', 'STC_10_12_26_0.5', 'STCmacd_10_12_26_0.5', 'STCstoch_10_12_26_0.5', 'STOCHk_14_3_3', 'STOCHd_14_3_3', 'STOCHh_14_3_3', 'STOCHFk_14_3', 'STOCHFd_14_3', 'STOCHRSIk_14_14_3_3', 'STOCHRSId_14_14_3_3', 'TRIX_30_9', 'TRIXs_30_9', 'TSI_13_25_13', 'TSIs_13_25_13', 'UO',

,model_key,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,n_folds,n_test_total
model,,,,,,,,,
LightGBM,lightgbm,0.7608,0.7349,0.6225,0.6493,0.6109,0.8393,4,7102
XGBoost,xgboost,0.7705,0.7521,0.6220,0.6896,0.6376,0.8399,4,7102
RandomForest,rf,0.6838,0.6263,0.4757,0.4721,0.4636,0.6857,4,7102



best model by fold-weighted ROC-AUC: 'xgboost'

fold-by-fold detail for the winner:


,fold,n_train,n_test,positive_rate,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,class_weight_0,class_weight_1,n
0,1,2340,1794,0.2971,0.6973,0.7040,0.4936,0.7205,0.5858,0.7841,1.0104,0.9898,1794
1,2,4691,1789,0.2174,0.7915,0.7953,0.5132,0.8021,0.6259,0.8676,0.8144,1.2951,1789
2,3,7006,1796,0.3764,0.7890,0.7469,0.8075,0.5769,0.6730,0.8487,0.7587,1.4663,1796
3,4,9359,1723,0.2954,0.8056,0.7628,0.6754,0.6582,0.6667,0.8600,0.7665,1.4381,1723



wrote data\artifacts\model_comparison_initial.json


## M13 — Boruta

In [17]:
!python -m ml_scan.cli ml boruta --in data/artifacts/labeled_smoke.parquet --out data/artifacts/boruta_features.json --max-iter 50

boruta (lightgbm) kept 67 features
wrote C:\Users\mail2\OneDrive\projects2\ml_scan\data\artifacts\boruta_features.json


### M13b — Boruta on the best model + modeling results (Q4)

Re-runs Boruta with `estimator_name=best_model` (the M12b winner) as the shadow-feature classifier — the M13 gate cell above always uses the configured default (`ml.model` in `configs/default.yaml`), which happens to be the same model here but will not always be. Reports how many of the initial features survive, and re-fits `best_model` on just those features so the effect of feature selection is visible before VIF runs.

In [18]:
import json
from pathlib import Path

import pandas as pd

from ml_scan.config import load_settings
from ml_scan.features.qc import select_xy
from ml_scan.ml_engine.estimator import compare_models
from ml_scan.ml_engine.metrics import fold_metrics_table
from ml_scan.ml_engine.selector import BorutaSelector
from ml_scan.ml_engine.splitter import PurgedWalkForward

settings = load_settings()
labeled = pd.read_parquet('data/artifacts/labeled_smoke.parquet')
init = json.loads(Path('data/artifacts/model_comparison_initial.json').read_text())
best_model, initial_features = init['best_model'], init['initial_features']

X, y, _ = select_xy(labeled, initial_features)
boruta = BorutaSelector(
    max_iter=settings.ml.boruta_max_iter,
    random_state=settings.ml.random_state,
    estimator_name=best_model,
).fit(X, y)
boruta.save('data/artifacts/boruta_features_best_model.json')
print(f"Q4 -- Boruta ({best_model}) kept {len(boruta.features_)} of {len(initial_features)} initial features:")
print(boruta.features_)

Xb, yb, panelb = select_xy(labeled, boruta.features_)
splitter = PurgedWalkForward(n_splits=settings.ml.n_splits, embargo_sessions=settings.ml.embargo_sessions)
boruta_summary, boruta_models = compare_models(
    Xb, yb, panelb, models=(best_model,), splitter=splitter, random_state=settings.ml.random_state
)
print("\nQ0/Q4 -- modeling results with Boruta-selected features:")
display(boruta_summary.round(4))
print("\nfold-by-fold detail:")
display(fold_metrics_table(boruta_models[best_model].fold_metrics_).round(4))

_ = Path('data/artifacts/model_comparison_boruta.json').write_text(
    json.dumps(
        {'n_features': len(boruta.features_), 'summary': boruta_summary.reset_index().to_dict(orient='records')},
        indent=2,
    ),
    encoding='utf-8',
)

Q4 -- Boruta (xgboost) kept 108 of 218 initial features:
['AO', 'BIAS', 'CCI', 'CFO', 'CMO', 'BULLP_13', 'BEARP_13', 'EXHC_DNa', 'EXHC_UPa', 'J_9_3', 'KST_10_15_20_30_10_10_10_15', 'KSTs_9', 'MACDh_12_26_9', 'MACDs_12_26_9', 'PGO', 'PPOs_12_26_9', 'PSL', 'QQE_14_5_4.236', 'QQE_14_5_4.236_RSIMA', 'SMI_5_20_5_1.0', 'SMIo_5_20_5_1.0', 'STC_10_12_26_0.5', 'STCstoch_10_12_26_0.5', 'STOCHFk_14_3', 'TRIX_30_9', 'TRIXs_30_9', 'TSI_13_25_13', 'TSIs_13_25_13', 'AGj_13_8_5', 'AGl_13_8_5', 'DEMA', 'FWMA', 'HL2', 'HLC3', 'KAMA', 'LINREG', 'MIDPOINT', 'MIDPRICE', 'SUPERT_7_3.0', 'VIDYA', 'LOG_RETURN', 'MAD', 'MEDIAN', 'SKEW', 'STDEV', 'TOS_STDEVALL_LR', 'TOS_STDEVALL_L_1', 'TOS_STDEVALL_U_1', 'TOS_STDEVALL_L_2', 'TOS_STDEVALL_U_2', 'TOS_STDEVALL_L_3', 'TOS_STDEVALL_U_3', 'ZSCORE', 'ADX_14', 'ADXR_14_2', 'DMN_14', 'AMATe_SR_8_21_2', 'CHOP', 'CKSPl_10_3_20', 'CKSPs_10_3_20', 'DECAY', 'DPO', 'HT_TRENDLINE', 'TRENDFLEX', 'VHF', 'VTXM_14', 'ABER_ATR_5_15', 'ACCBL_20', 'ACCBM_20', 'ACCBU_20', 'ATR', 'ATRT

,model_key,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,n_folds,n_test_total
model,,,,,,,,,
XGBoost,xgboost,0.7671,0.7515,0.6194,0.6928,0.6343,0.8393,4,7102



fold-by-fold detail:


,fold,n_train,n_test,positive_rate,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,class_weight_0,class_weight_1,n
0,1,2340,1794,0.2971,0.7046,0.7124,0.5019,0.7317,0.5954,0.7775,1.0104,0.9898,1794
1,2,4691,1789,0.2174,0.7680,0.7803,0.4800,0.8021,0.6006,0.8578,0.8144,1.2951,1789
2,3,7006,1796,0.3764,0.7867,0.7413,0.8178,0.5577,0.6631,0.8567,0.7587,1.4663,1796
3,4,9359,1723,0.2954,0.8108,0.7727,0.6798,0.6798,0.6798,0.8661,0.7665,1.4381,1723


## M14 — VIF

In [19]:
!python -m ml_scan.cli ml vif --in data/artifacts/labeled_smoke.parquet --features data/artifacts/boruta_features.json --out data/artifacts/selected_features.json --max-vif 10

vif kept 44 features
wrote C:\Users\mail2\OneDrive\projects2\ml_scan\data\artifacts\selected_features.json


### M14b — Modeling results after VIF pruning (Q5 rationale + Q6)

**Why VIF, and why now**: Boruta already removes pure-noise columns; most of what survives is still overlapping transforms of the same handful of price/volume series (three Bollinger Band columns, `ATR` vs. `ATRr_14`, `RSI_14` vs. `MFI_14`, ...). Variance Inflation Factor pruning is the right second pass for exactly this failure mode: unlike a plain pairwise-correlation filter, VIF measures how well *each* remaining column is predicted by *all the others combined*, so it catches multi-column redundancy a correlation matrix alone would miss. `VIFPruner` (`ml_engine/selector.py`) also pre-drops any pair correlated above 0.95 before computing VIF, which keeps the regression well-conditioned. Fewer, less-redundant columns mean the tree models spend their limited splits on distinct signals instead of several near-copies of the same one.

The cell below re-fits the M12b winner on the final `selected_features.json` list (Boruta → VIF) and lines the results up against M12b (initial) and M13b (Boruta) so the effect of each stage is visible side by side.

In [20]:
import json
from pathlib import Path

import pandas as pd

from ml_scan.config import load_settings
from ml_scan.features.qc import select_xy
from ml_scan.ml_engine.estimator import compare_models
from ml_scan.ml_engine.metrics import fold_metrics_table
from ml_scan.ml_engine.selector import load_feature_list
from ml_scan.ml_engine.splitter import PurgedWalkForward

settings = load_settings()
labeled = pd.read_parquet('data/artifacts/labeled_smoke.parquet')
init = json.loads(Path('data/artifacts/model_comparison_initial.json').read_text())
best_model, initial_features = init['best_model'], init['initial_features']
boruta_info = json.loads(Path('data/artifacts/model_comparison_boruta.json').read_text())
selected = load_feature_list('data/artifacts/selected_features.json')

print(f"Q5 -- feature counts by stage: initial={len(initial_features)} -> boruta={boruta_info['n_features']} -> vif={len(selected)}")
print(selected)

Xs, ys, panels = select_xy(labeled, selected)
splitter = PurgedWalkForward(n_splits=settings.ml.n_splits, embargo_sessions=settings.ml.embargo_sessions)
vif_summary, vif_models = compare_models(
    Xs, ys, panels, models=(best_model,), splitter=splitter, random_state=settings.ml.random_state
)
print("\nQ0/Q6 -- modeling results with VIF-selected features:")
display(vif_summary.round(4))
print("\nfold-by-fold detail:")
display(fold_metrics_table(vif_models[best_model].fold_metrics_).round(4))

stages = pd.DataFrame(
    [
        {'stage': f"initial ({len(initial_features)} features)", **next(r for r in init['summary'] if r['model_key'] == best_model)},
        {'stage': f"boruta ({boruta_info['n_features']} features)", **boruta_info['summary'][0]},
        {'stage': f"vif ({len(selected)} features)", **vif_summary.reset_index().to_dict(orient='records')[0]},
    ]
).set_index('stage')[['accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1', 'roc_auc']]
print("\nQ6 -- selection-stage comparison (same model, shrinking feature set):")
display(stages.round(4))

_ = Path('data/artifacts/model_comparison_vif.json').write_text(
    json.dumps({'n_features': len(selected), 'summary': vif_summary.reset_index().to_dict(orient='records')}, indent=2),
    encoding='utf-8',
)

Q5 -- feature counts by stage: initial=218 -> boruta=108 -> vif=44
['CCI', 'BULLP_13', 'FISHERTs_9_1', 'INERTIA', 'J_9_3', 'KSTs_9', 'MACDs_12_26_9', 'SMIo_5_20_5_1.0', 'STC_10_12_26_0.5', 'TRIX_30_9', 'HILOs_13_21', 'SUPERTs_7_3.0', 'KURTOSIS', 'SKEW', 'TOS_STDEVALL_L_2', 'ADX_14', 'DMP_14', 'DMN_14', 'CHOP', 'DPO', 'VHF', 'BBB_5_2.0_2.0', 'BBP_5_2.0_2.0', 'HWW_1', 'HWPCT_1', 'MASSI', 'NATR', 'RVI', 'UI', 'AD', 'ADOSC', 'OBV', 'CMF', 'EFI', 'EOM', 'KVOs_34_55_13', 'MFI', 'NVI', 'PVIe_255', 'PVOs_12_26_9', 'PVT', 'TSVs_18_10', 'd_rs', 'm15_atr']

Q0/Q6 -- modeling results with VIF-selected features:


,model_key,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,n_folds,n_test_total
model,,,,,,,,,
XGBoost,xgboost,0.765,0.7445,0.604,0.6862,0.6323,0.8369,4,7102



fold-by-fold detail:


,fold,n_train,n_test,positive_rate,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,class_weight_0,class_weight_1,n
0,1,2340,1794,0.2971,0.6856,0.6913,0.4802,0.7054,0.5714,0.7788,1.0104,0.9898,1794
1,2,4691,1789,0.2174,0.8105,0.7694,0.5508,0.6967,0.6152,0.8533,0.8144,1.2951,1789
2,3,7006,1796,0.3764,0.7890,0.7516,0.7883,0.6006,0.6818,0.8703,0.7587,1.4663,1796
3,4,9359,1723,0.2954,0.7754,0.7665,0.5959,0.7446,0.6620,0.8456,0.7665,1.4381,1723



Q6 -- selection-stage comparison (same model, shrinking feature set):


,accuracy,balanced_accuracy,precision,recall,f1,roc_auc
stage,,,,,,
initial (218 features),0.7705,0.7521,0.6220,0.6896,0.6376,0.8399
boruta (108 features),0.7671,0.7515,0.6194,0.6928,0.6343,0.8393
vif (44 features),0.7650,0.7445,0.6040,0.6862,0.6323,0.8369


## M15 — Purged walk-forward

In [21]:
!python -m pytest tests/test_purged_split.py -q

.                                                                        [100%]
1 passed in 0.05s


### M15b — Which stocks land in each train/test fold (Q1, completed)

The split used everywhere in this notebook is **time-based, not stock-based**: `PurgedWalkForward` slices the timeline into ordered blocks with an embargo gap between them, so every stock with rows inside a given window eventually appears on *both* sides — there is no fixed "training stocks" vs. "held-out stocks" split, only "earlier time" (train) vs. "later time, after the embargo" (test) per fold. The table below shows, fold by fold, exactly which smoke-universe stocks contributed train rows vs. test rows, and how many.

In [22]:
import pandas as pd

from ml_scan.config import load_settings
from ml_scan.features.qc import select_xy, symbol_row_counts
from ml_scan.ml_engine.selector import load_feature_list
from ml_scan.ml_engine.splitter import PurgedWalkForward, fold_symbol_table

settings = load_settings()
labeled = pd.read_parquet('data/artifacts/labeled_smoke.parquet')
selected = load_feature_list('data/artifacts/selected_features.json')
_, _, panel = select_xy(labeled, selected)
splitter = PurgedWalkForward(n_splits=settings.ml.n_splits, embargo_sessions=settings.ml.embargo_sessions)

print("Q1 -- stocks fed into the model, with resolved-label row counts per stock:")
display(symbol_row_counts(labeled))

fold_table = fold_symbol_table(panel, splitter)
pivot = (
    fold_table.pivot_table(index='symbol', columns='fold', values=['n_train_rows', 'n_test_rows'], aggfunc='sum')
    .fillna(0)
    .astype(int)
)
print(f"\n{fold_table['symbol'].nunique()} stocks appear across {fold_table['fold'].nunique()} purged walk-forward folds; rows per stock per fold (train vs test):")
display(pivot)

Q1 -- stocks fed into the model, with resolved-label row counts per stock:


,symbol,n_rows,n_win,n_loss,n_unresolved,win_rate
0,BHARTIARTL,1496,449,1006,41,0.308591
1,BSE,1496,541,925,30,0.369031
2,ETERNAL,1496,484,973,39,0.332189
3,HDFCBANK,1496,355,1099,42,0.244154
4,ICICIBANK,1496,502,945,49,0.346925
5,NETWEB,1496,524,918,54,0.363384
6,RELIANCE,1496,419,1034,43,0.288369
7,SBIN,1496,684,782,30,0.466576



8 stocks appear across 4 purged walk-forward folds; rows per stock per fold (train vs test):


n_test_rows                n_train_rows                
fold                 1    2    3    4            1    2    3     4
symbol                                                            
BHARTIARTL         226  225  226  213          296  592  876  1172
BSE                220  226  226  223          292  581  877  1173
ETERNAL            224  226  226  210          291  585  881  1177
HDFCBANK           226  226  219  218          289  583  879  1166
ICICIBANK          224  226  226  208          296  590  873  1169
NETWEB             226  216  226  210          289  585  868  1164
RELIANCE           226  221  224  216          292  588  873  1167
SBIN               222  223  223  225          295  587  879  1171

## M16 — Train LightGBM

Display fold metrics table.

In [23]:
!python -m ml_scan.cli ml train --in data/artifacts/labeled_smoke.parquet --features data/artifacts/selected_features.json --out data/artifacts/model.joblib
import json, pandas as pd
from pathlib import Path
meta = json.loads(Path('data/artifacts/model.json').read_text())
display(pd.DataFrame(meta.get('fold_metrics', [])))

[
  {
    "fold": 1,
    "n_train": 2340,
    "n_test": 1794,
    "class_weight_0": 1.0103626943005182,
    "class_weight_1": 0.9898477157360406,
    "n": 1794,
    "positive_rate": 0.2971014492753623,
    "accuracy": 0.701783723522854,
    "precision": 0.49874686716791977,
    "recall": 0.7467166979362101,
    "f1": 0.5980465815176559,
    "balanced_accuracy": 0.7147540666524825,
    "roc_auc": 0.7939461072766038
  },
  {
    "fold": 2,
    "n_train": 4691,
    "n_test": 1789,
    "class_weight_0": 0.8144097222222223,
    "class_weight_1": 1.2951408061844285,
    "n": 1789,
    "positive_rate": 0.21743991056456122,
    "accuracy": 0.7931805477920626,
    "precision": 0.5185185185185185,
    "recall": 0.6838046272493573,
    "f1": 0.5898004434589801,
    "balanced_accuracy": 0.753688027910393,
    "roc_auc": 0.8451762761659934
  },
  {
    "fold": 3,
    "n_train": 7006,
    "n_test": 1796,
    "class_weight_0": 0.758717782109595,
    "class_weight_1": 1.4663038928421934,
    "n": 1796

,fold,n_train,n_test,class_weight_0,class_weight_1,n,positive_rate,accuracy,precision,recall,f1,balanced_accuracy,roc_auc
0,1,2340,1794,1.010363,0.989848,1794,0.297101,0.701784,0.498747,0.746717,0.598047,0.714754,0.793946
1,2,4691,1789,0.814410,1.295141,1789,0.217440,0.793181,0.518519,0.683805,0.589800,0.753688,0.845176
2,3,7006,1796,0.758718,1.466304,1796,0.376392,0.772829,0.787554,0.542899,0.642732,0.727253,0.859577
3,4,9359,1723,0.766503,1.438076,1723,0.295415,0.778294,0.601276,0.740668,0.663732,0.767369,0.851712


### M16a — Hyperparameter optimization (Q7)

**Method: randomized search** (`sklearn.model_selection.RandomizedSearchCV`), scored on the *same* purged, embargoed walk-forward folds used everywhere in this notebook — the folds are passed in directly as `cv=[(train_idx, test_idx), ...]`, so the search cannot reward parameters that only look good on shuffled or leaky folds (sklearn's default K-fold would do exactly that on time-ordered data). The search space (`ml_engine/optimize.py::PARAM_DISTRIBUTIONS`) covers tree depth, learning rate, tree count, subsampling, and regularization for whichever model won M12b.

In [24]:
import json
from pathlib import Path

import pandas as pd

from ml_scan.config import load_settings
from ml_scan.features.qc import select_xy
from ml_scan.ml_engine.optimize import random_search
from ml_scan.ml_engine.selector import load_feature_list
from ml_scan.ml_engine.splitter import PurgedWalkForward

settings = load_settings()
labeled = pd.read_parquet('data/artifacts/labeled_smoke.parquet')
init = json.loads(Path('data/artifacts/model_comparison_initial.json').read_text())
best_model = init['best_model']
selected = load_feature_list('data/artifacts/selected_features.json')

Xs, ys, panels = select_xy(labeled, selected)
splitter = PurgedWalkForward(n_splits=settings.ml.n_splits, embargo_sessions=settings.ml.embargo_sessions)
best_params, search_results = random_search(
    Xs, ys, panels, model_name=best_model, splitter=splitter, n_iter=20, random_state=settings.ml.random_state
)
print(f"Q7 -- optimization method: RandomizedSearchCV on purged walk-forward folds, model={best_model!r}")
print("best_params:")
print(json.dumps(best_params, indent=2, default=str))

Path('data/artifacts/best_params.json').write_text(
    json.dumps({'model_name': best_model, 'best_params': best_params}, indent=2, default=str), encoding='utf-8'
)
print("\ntop parameter combinations tried, by mean ROC-AUC across folds:")
display(search_results.head(10))

Q7 -- optimization method: RandomizedSearchCV on purged walk-forward folds, model='xgboost'
best_params:
{
  "subsample": 0.6,
  "reg_lambda": 5.0,
  "n_estimators": 300,
  "min_child_weight": 5,
  "max_depth": 5,
  "learning_rate": 0.02,
  "colsample_bytree": 0.6
}

top parameter combinations tried, by mean ROC-AUC across folds:


,param_subsample,param_reg_lambda,param_n_estimators,param_min_child_weight,param_max_depth,param_learning_rate,param_colsample_bytree,mean_test_score,std_test_score,rank_test_score
0,0.6,5.0,300,5,5,0.02,0.6,0.845259,0.030661,1
1,0.7,0.5,200,5,6,0.03,0.6,0.844786,0.030343,2
2,0.8,2.0,150,10,4,0.08,0.6,0.842582,0.026870,3
3,0.6,5.0,200,3,4,0.10,0.7,0.839832,0.026283,4
4,0.7,0.5,300,3,5,0.02,0.6,0.839805,0.034201,5
5,0.7,2.0,150,5,6,0.10,0.6,0.839038,0.026405,6
6,1.0,1.0,150,1,4,0.03,0.8,0.837863,0.032996,7
7,1.0,0.5,100,5,6,0.08,0.7,0.836913,0.026467,8
8,0.8,5.0,150,10,3,0.10,0.6,0.836760,0.033272,9
9,1.0,1.0,150,1,5,0.05,0.8,0.835191,0.034638,10


### M16b — Retrain with the tuned hyperparameters (Q8, final model)

This is the model the rest of the notebook (M17 scan, M20/M21 backtest) actually uses from here on: the M12b winner retrained on the VIF-selected features with the hyperparameters found in M16a, saved over `data/artifacts/model.joblib`. Reported with the full metric set, fold by fold and as a fold-weighted average, so it is directly comparable to M12b/M13b/M14b above.

In [25]:
import json
from pathlib import Path

import pandas as pd

from ml_scan.config import load_settings
from ml_scan.features.qc import select_xy
from ml_scan.ml_engine.artifacts import save_model
from ml_scan.ml_engine.estimator import MLEstimator
from ml_scan.ml_engine.metrics import fold_metrics_table
from ml_scan.ml_engine.selector import load_feature_list
from ml_scan.ml_engine.splitter import PurgedWalkForward

settings = load_settings()
labeled = pd.read_parquet('data/artifacts/labeled_smoke.parquet')
init = json.loads(Path('data/artifacts/model_comparison_initial.json').read_text())
best_model = init['best_model']
best_params = json.loads(Path('data/artifacts/best_params.json').read_text())['best_params']
selected = load_feature_list('data/artifacts/selected_features.json')

Xs, ys, panels = select_xy(labeled, selected)
splitter = PurgedWalkForward(n_splits=settings.ml.n_splits, embargo_sessions=settings.ml.embargo_sessions)
tuned = MLEstimator(model=best_model, random_state=settings.ml.random_state, params=best_params)
tuned.fit_walk_forward(Xs, ys, panels, splitter)

print(f"Q8 -- final tuned {best_model} fold metrics:")
display(fold_metrics_table(tuned.fold_metrics_).round(4))
print("\nfold-weighted average:")
print(json.dumps(tuned.aggregate_metrics(), indent=2))

dest = save_model(tuned, 'data/artifacts/model.joblib')
print(f"\nwrote {dest} -- M17 (scan) and M20/M21 (backtest, Q9) below now run on this tuned model.")

Q8 -- final tuned xgboost fold metrics:


,fold,n_train,n_test,positive_rate,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,class_weight_0,class_weight_1,n
0,1,2340,1794,0.2971,0.7040,0.6958,0.5014,0.6754,0.5755,0.7952,1.0104,0.9898,1794
1,2,4691,1789,0.2174,0.7954,0.7690,0.5213,0.7224,0.6056,0.8566,0.8144,1.2951,1789
2,3,7006,1796,0.3764,0.7851,0.7470,0.7832,0.5932,0.6751,0.8625,0.7587,1.4663,1796
3,4,9359,1723,0.2954,0.7998,0.7877,0.6349,0.7583,0.6911,0.8745,0.7665,1.4381,1723



fold-weighted average:
{
  "accuracy": 0.7707687975218248,
  "balanced_accuracy": 0.7495037577467518,
  "precision": 0.6100646955827429,
  "recall": 0.6865718968631772,
  "f1": 0.6363309968030123,
  "roc_auc": 0.8469262604073245,
  "n_folds": 4,
  "n_test_total": 7102
}

wrote data\artifacts\model.joblib -- M17 (scan) and M20/M21 (backtest, Q9) below now run on this tuned model.


## M17 — Inference scan

In [26]:
!python -m ml_scan.cli scan --asof latest --universe data/universe/smoke_symbols.csv --out data/artifacts/scan_latest.csv
import pandas as pd
scan = pd.read_csv('data/artifacts/scan_latest.csv')
display(scan)

wrote C:\Users\mail2\OneDrive\projects2\ml_scan\data\artifacts\scan_latest.csv


,symbol,asof_ts,score,p_win,entry_px,sl_px,tp_px,atr,adtv_20,rank
0,BSE,2026-09-03 14:15:00+05:30,0.793452,0.793452,3306.10,3235.672710,3446.954580,35.213645,NaN,1
1,NETWEB,2026-09-03 14:15:00+05:30,0.722271,0.722271,5270.00,5121.851365,5566.297271,74.074318,NaN,2
2,HDFCBANK,2026-09-03 14:15:00+05:30,0.645848,0.645848,708.75,699.991035,726.267929,4.379482,NaN,3
3,SBIN,2026-09-03 14:15:00+05:30,0.416454,0.416454,1023.00,1011.176624,1046.646752,5.911688,NaN,4
4,BHARTIARTL,2026-09-03 14:15:00+05:30,0.406304,0.406304,1865.30,1847.436483,1901.027034,8.931758,NaN,5
5,RELIANCE,2026-09-03 14:15:00+05:30,0.382793,0.382793,1309.00,1295.413529,1336.172941,6.793235,NaN,6
6,ICICIBANK,2026-09-03 14:15:00+05:30,0.299561,0.299561,1432.30,1416.159071,1464.581858,8.070464,NaN,7
7,ETERNAL,2026-09-03 14:15:00+05:30,0.250249,0.250249,326.45,320.350343,338.649315,3.049829,NaN,8


## M18 — Indian costs

In [27]:
!python -m pytest tests/test_costs.py -q

.                                                                        [100%]
1 passed in 0.14s


## M19 — Next-open fill lag

In [28]:
!python -m pytest tests/test_fill_lag.py -q

.                                                                        [100%]
1 passed in 0.05s


## M20 — Event-driven backtest

In [29]:
import subprocess, sys
from pathlib import Path
import pandas as pd
from ml_scan.config import load_settings
from ml_scan.execution.scanner import InferenceScanner, history_signals

settings = load_settings()
lab = pd.read_parquet('data/artifacts/labeled_smoke.parquet')
scanner = InferenceScanner(settings)
scanner.load_artifacts('data/artifacts/model.joblib', 'data/artifacts/selected_features.json')
hist = history_signals(scanner.score_panel(lab), threshold=settings.ml.score_threshold)
Path('data/artifacts/scan_history.parquet').parent.mkdir(parents=True, exist_ok=True)
hist.to_parquet('data/artifacts/scan_history.parquet', index=False)
print('scan_history rows', len(hist))

subprocess.check_call([
    sys.executable, "-m", "ml_scan.cli", "backtest", "run",
    "--signals", "data/artifacts/scan_history.parquet",
    "--start", SMOKE_START, "--end", SMOKE_END,
    "--out", "data/artifacts/bt_smoke",
])

scan_history rows 4334


0

## M21 — Metrics

In [30]:
!python -m ml_scan.cli backtest metrics --run data/artifacts/bt_smoke
import pandas as pd
from ml_scan.reporting.charts import equity_figure
eq = pd.read_parquet('data/artifacts/bt_smoke/equity.parquet')
fig = equity_figure(eq)
fig.show()

{
  "cagr": 2.4941348386855826,
  "sharpe": 7.549273165326652,
  "sortino": 14.359839790784894,
  "max_dd": -0.03220028245070894,
  "win_rate": 0.7447916666666666,
  "profit_factor": 4.262434319070705,
  "n_trades": 384.0,
  "avg_r": 1.20140878108754,
  "exposure": 0.0
}


## M22 — Reporting

In [31]:
!python -m ml_scan.cli report --run data/artifacts/bt_smoke --out data/artifacts/report_smoke.html
from ml_scan.ml_engine.artifacts import load_model
from ml_scan.reporting.charts import importance_figure
imps = load_model('data/artifacts/model.joblib').feature_importances()
if not imps.empty:
    importance_figure(imps).show()

wrote data\artifacts\report_smoke.html


## M23 — CLI surface

In [32]:
!python -m ml_scan.cli --help

                                                                               
 Usage: python -m ml_scan.cli [OPTIONS] COMMAND [ARGS]...                      
                                                                               
 Nifty 500 MTF ML scanner                                                      
                                                                               
┌─ Options ───────────────────────────────────────────────────────────────────┐
│ --install-completion          Install completion for the current shell.     │
│ --show-completion             Show completion for the current shell, to     │
│                               copy it or customize the installation.        │
│ --help                        Show this message and exit.                   │
└─────────────────────────────────────────────────────────────────────────────┘
┌─ Commands ──────────────────────────────────────────────────────────────────┐
│ coverage                              

## M24 — End-to-end smoke (run everything)

In [33]:
!python -m ml_scan.cli e2e smoke

y
0.0    1826
1.0     969
NaN     229
y_reason
sl_hit        1826
tp_hit         969
unresolved     229
{
  "n_rows": 3024,
  "n_feature_cols": 213,
  "leakage_ok": true
}
boruta (lightgbm) kept 37 features
vif kept 30 features
[
  {
    "fold": 1,
    "n_train": 570,
    "n_test": 427,
    "class_weight_0": 1.0,
    "class_weight_1": 1.0,
    "n": 427,
    "positive_rate": 0.3185011709601874,
    "accuracy": 0.6932084309133489,
    "precision": 0.5308641975308642,
    "recall": 0.3161764705882353,
    "f1": 0.39631336405529954,
    "balanced_accuracy": 0.5927961390741864,
    "roc_auc": 0.7437588437436831
  },
  {
    "fold": 2,
    "n_train": 1141,
    "n_test": 424,
    "class_weight_0": 0.8696646341463415,
    "class_weight_1": 1.1762886597938145,
    "n": 424,
    "positive_rate": 0.33726415094339623,
    "accuracy": 0.7051886792452831,
    "precision": 0.5714285714285714,
    "recall": 0.5034965034965035,
    "f1": 0.5353159851301115,
    "balanced_accuracy": 0.6556628424955827,


C:\Users\mail2\OneDrive\projects2\ml_scan\src\ml_scan\features\htf.py:92: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(parts, ignore_index=True)


## M25 — Liquid-universe scale-up

Same pipeline on up to 80 liquid names. With full pandas_ta this is a long run (Boruta + VIF on ~100k rows); the code cell is tagged skip-execute so a full notebook restart stays on the smoke path. Run it from the CLI when you want the scale-up:

python -m ml_scan.cli e2e liquid --max-symbols 80

In [34]:
!python -m ml_scan.cli e2e liquid --max-symbols 80

y
0.0    72609
1.0    41238
NaN     4263
y_reason
sl_hit        72609
tp_hit        41238
unresolved     4263
boruta (lightgbm) kept 103 features
vif kept 60 features
[
  {
    "fold": 1,
    "n_train": 21837,
    "n_test": 17867,
    "class_weight_0": 0.8848055105348459,
    "class_weight_1": 1.1496788459513532,
    "n": 17867,
    "positive_rate": 0.3285946157720938,
    "accuracy": 0.7896121341019757,
    "precision": 0.6724363161332463,
    "recall": 0.7014137284959973,
    "f1": 0.6866194247603168,
    "balanced_accuracy": 0.7670956605134205,
    "roc_auc": 0.8597647861840508
  },
  {
    "fold": 2,
    "n_train": 45162,
    "n_test": 17944,
    "class_weight_0": 0.7965079365079364,
    "class_weight_1": 1.3431477516059958,
    "n": 17944,
    "positive_rate": 0.31525858225590725,
    "accuracy": 0.7974253232278199,
    "precision": 0.643485665625887,
    "recall": 0.8014848859819692,
    "f1": 0.7138471227269149,
    "balanced_accuracy": 0.7985205824880139,
    "roc_auc": 0.88446

C:\Users\mail2\OneDrive\projects2\ml_scan\src\ml_scan\features\htf.py:92: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(parts, ignore_index=True)
